In [ ]:
# Instalujemy DuckDB. To dosłownie zajmuje 2 sekundy! Brak JVM, brak serwerów w tle.
!pip install -q duckdb pandas pyarrow
print("DuckDB zainstalowany i gotowy do walki!")

### 1. Klasyczne (bolesne) podejście: Pandas
Najpierw spróbujmy zrobić to tak, jak robiło się to przez ostatnią dekadę. 
Użyjemy biblioteki `pandas`. Musimy wczytać CAŁY plik Parquet z chmury do pamięci operacyjnej (RAM), a dopiero potem pogrupować.

In [ ]:
import pandas as pd
import time

# Używamy publicznego, 40-megabajtowego pliku z logami taksówek (prawie 3 miliony wierszy)
parquet_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"

print("Próba Pandas: Wczytuję plik z sieci do pamięci RAM... (to potrwa i zużyje dużo pamięci)")
start_time = time.time()

# Pandas musi zdeserializować każdą kolumnę
df_pandas = pd.read_parquet(parquet_url)

# Liczymy sumę zysków z podziałem na identyfikator strefy odbioru pasażera
result_pandas = df_pandas.groupby('PULocationID')['total_amount'].sum().sort_values(ascending=False).head(5)

end_time = time.time()
time_pandas = end_time - start_time

print(result_pandas)
print(f"\nPandas pracował: {time_pandas:.2f} sekundy")

### 2. Nowa Epoka: DuckDB + SQL
Teraz wjeżdża DuckDB. Napiszemy kod SQL, który celuje BEZPOŚREDNIO w link URL (pliku w chmurze AWS).

Kluczowe mechanizmy:
1. **Wektoryzacja (SIMD):** DuckDB użyje pełnej mocy procesora do grupowania.
2. **Pushdown & Pruning:** DuckDB zobaczy w zapytaniu `PULocationID` i `total_amount` i zignoruje resztę z 19 kolumn w pliku Parquet! Nawet nie pobierze ich z internetu!

In [ ]:
import duckdb

print("Próba DuckDB: Odpytywanie strumieniowe pliku z S3 w chmurze...")
start_time = time.time()

# Zobacz jak genialnie prosty jest ten SQL. Adres URL traktujemy jak tabelę!
query = f"""
    SELECT 
        PULocationID, 
        SUM(total_amount) as total_revenue
    FROM '{parquet_url}'
    GROUP BY PULocationID
    ORDER BY total_revenue DESC
    LIMIT 5
"""

# Odpalamy zapytanie i konwertujemy wynik od razu na ładny obiekt Pandas (.df())
result_duckdb = duckdb.sql(query).df()

end_time = time.time()
time_duckdb = end_time - start_time

print(result_duckdb)
print(f"\nDuckDB rozwiązał problem w: {time_duckdb:.2f} sekundy!")

### 3. Zderzenie Światów (Zero-Copy Integration)
Co jeśli mamy skomplikowany, wyczyszczony obiekt Pandas (DataFrame) z setkami tysięcy wierszy i nagle musimy napisać na nim skomplikowanego SQL-a (bo numpy/pandas jest zbyt trudny)?

W DuckDB możesz **bezpośrednio wpisać nazwę zmiennej Pythona do klauzuli FROM w SQL-u!** Kopiowanie pamięci wynosi równe **Zero**.

In [ ]:
import numpy as np

# Tworzymy gigantyczną "brudną" tabelę w czystym Pythonie (10 milionów wierszy)
print("Tworzę 10 milionów wierszy (Zmienna 'dirty_dataframe')...")
dirty_dataframe = pd.DataFrame({
    'user': np.random.randint(1, 100, 10000000),
    'score': np.random.randn(10000000)
})

# Chcemy wyciągnąć top 3 użytkowników o największym średnim wyniku.
# W Pandasie składnia bywa zawiła. Zróbmy to w czystym SQL!

sql = """
    SELECT user, AVG(score) as srednia 
    FROM dirty_dataframe  -- Używamy bezpośrednio nazwy zmiennej Pythonowej!
    GROUP BY user 
    ORDER BY srednia DESC 
    LIMIT 3
"""

print("\nOdpytuję zmienną Pythona za pomocą SQL (DuckDB)...")
start_time = time.time()

# duckdb.query() automatycznie skanuje zmienne globalne w poszukiwaniu 'dirty_dataframe'
final_result = duckdb.query(sql).df()

end_time = time.time()

print(final_result)
print(f"\nPrzetworzenie 10 milionów wierszy ze zmiennej w RAMie zajęło DuckDB: {end_time - start_time:.3f} s!")